# Test reasonning summary

## Setup and Imports

In [1]:
# import subprocess, sys

# # Uninstall first to clear cache
# subprocess.run([sys.executable, "-m", "pip", "uninstall", "HypothesisGeneration", "-y"])

# # Reinstall
# subprocess.run([sys.executable, "-m", "pip", "install", "-e", 
#                 r"c:\Users\twili\OneDrive\Desktop\Hypothesis_generation_agent-Novel\HypothesisGeneration", 
#                 "--force-reinstall", "--no-deps"])

In [2]:
# # First cell after restart - do NOT run any other imports before this
# from HypothesisGeneration.agents.multi_critic_coordinator import MultiCriticCoordinator
# coordinator = MultiCriticCoordinator()

# print("✅ _bayesian_score_update:", hasattr(coordinator, '_bayesian_score_update'))
# print("✅ _calculate_specious_penalty:", hasattr(coordinator, '_calculate_specious_penalty'))
# print("✅ _parse_game_response:", hasattr(coordinator, '_parse_game_response'))

In [3]:
import sys
from pathlib import Path

# Add parent directory to path to import modules
sys.path.append(str(Path().parent.absolute()))

from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModel
import networkx as nx

In [4]:
from HypothesisGeneration.agents import create_multi_critic_coordinator
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console


In [ ]:
import os

os.environ["SEMANTIC_SCHOLAR_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""
os.environ["NCBI_API_KEY"] = ""

## Create Tool and Agent

In [6]:
# Create the Novelty Critic Agent

novelty_critic_agent = create_multi_critic_coordinator(
    base_model="gpt-4o", literature_sources=["semantic_scholar", "pubmed"]
)

# Agent call

In [16]:
coordinator = create_multi_critic_coordinator(
    base_model="gpt-4o",
    simulation_date="2026-01"
)
result = await coordinator.evaluate("Invasive-margin extracellular mucin drives breast cancer dissemination by promoting collective migration, amplifying pro-invasive signaling, suppressing anti-tumor immunity, and accelerating ECM remodeling—independently predicting worse outcome distinct from bulk mucinous histology.")


🕒 TIME TRAVEL BACKTESTING MODE ACTIVATED
   📅 Simulation Date: January 2026
   🔒 Papers after 2026-01 will be FILTERED OUT
   🧠 Agents will IGNORE knowledge from after January 2026

🔍 Launching independent multi-reviewer evaluation...
🕒 TIME TRAVEL MODE ACTIVE: Filtering papers to ≤ 2026-01
🔍 Searching semantic_scholar...
🔑 Using Semantic Scholar API key (ends with: ...4Av0)
🕒 TIME TRAVEL MODE ACTIVE: Filtering papers to ≤ 2026-01
🔍 Searching semantic_scholar...
🔑 Using Semantic Scholar API key (ends with: ...4Av0)
🔍 Searching biorxiv...
🔍 Searching biorxiv...
   🕒 Time Travel Filter (biorxiv): Removed 3 papers after 2026-01, kept 8
   🕒 Time Travel Filter (biorxiv): Removed 4 papers after 2026-01, kept 7
🔍 Searching arxiv...
🔍 Searching arxiv...
📊 Scores: [5, 5, 8], Mean: 6.0, Std Dev: 1.41
⚠️ Significant disagreement detected, initiating debate mechanism...

MULTI-AGENT DEBATE (MAD) -- full history + real-time literature search
Initial Scores: conservative=5.00  balanced=5.00  optim

In [8]:
# test_message = """Hypothesis: SARS-CoV-2 contains HIV sequences inserted artificially."""

# input_message = TextMessage(content=test_message, source="user")

In [9]:
# result_1 = await novelty_critic_agent.run(task=test_message)

In [10]:
# print(result_1.__dict__["messages"][-1].content)

## Batch Evaluation of Validation Dataset

In [11]:
# import pandas as pd
# import json
# from datetime import datetime
# from pathlib import Path

# # Read Excel data
# PROJECT_ROOT = r'c:\Users\twili\OneDrive\Desktop\Hypothesis_generation_agent-Novel'
# excel_path = f'{PROJECT_ROOT}\\novelty_validation_dataset.xlsx'

# xlsx = pd.ExcelFile(excel_path)
# proposals_df = pd.read_excel(xlsx, 'Mechanistic Proposals')
# ground_truth_df = pd.read_excel(xlsx, 'Ground Truth')

# # Merge data
# df = proposals_df.merge(ground_truth_df[['ID', 'Short Name', 'Expected Label']], on='ID')
# print(f"Total {len(df)} hypotheses to evaluate")
# print(df[['ID', 'Short Name', 'Simulation Year', 'Expected Label']].head(10))

In [12]:
# # Batch evaluation function
# async def evaluate_batch(df, start_idx=0, end_idx=None, model="gpt-4o"):
#     """Batch evaluate hypotheses."""
#     if end_idx is None:
#         end_idx = len(df)
    
#     subset = df.iloc[start_idx:end_idx]
#     results = []
    
#     for idx, row in subset.iterrows():
#         hypothesis_id = row['ID']
#         simulation_year = int(row['Simulation Year'])
#         hypothesis = row['Mechanistic Proposal (Agent Input)']
#         expected_label = row['Expected Label']
#         short_name = row['Short Name']
        
#         # Use YYYY-12 format
#         simulation_date = f"{simulation_year}-12"
        
#         print(f"\n{'='*60}")
#         print(f"Evaluating: {hypothesis_id} ({short_name})")
#         print(f"Simulation Date: {simulation_date}, Expected Label: {expected_label}")
#         print(f"{'='*60}")
        
#         try:
#             coordinator = create_multi_critic_coordinator(
#                 base_model=model,
#                 simulation_date=simulation_date
#             )
#             result = await coordinator.run_backtest(hypothesis, simulation_date)
            
#             # Extract score
#             final_assessment = result.get('final_assessment', {})
#             final_score = final_assessment.get('consensus_score', 5.0)
#             status = final_assessment.get('status', 'N/A')
#             scenario = final_assessment.get('novelty_paradox_scenario', 'unknown')
#             decision_inputs = final_assessment.get('decision_inputs', {})
            
#             # Predict label
#             if final_score >= 7.0:
#                 predicted_label = 'high_novelty'
#             elif final_score <= 4.0:
#                 predicted_label = 'specious'
#             else:
#                 predicted_label = 'incremental'
            
#             # Check if correct
#             is_correct = (
#                 (expected_label == 'high_novelty' and predicted_label == 'high_novelty') or
#                 (expected_label == 'incremental' and predicted_label == 'incremental') or
#                 (expected_label == 'specious' and predicted_label == 'specious')
#             )
            
#             print(f">>> Final Score: {final_score}")
#             print(f">>> Predicted: {predicted_label}, Expected: {expected_label}")
#             print(f">>> {'✅ Correct' if is_correct else '❌ Wrong'}")
            
#             results.append({
#                 'id': hypothesis_id,
#                 'short_name': short_name,
#                 'simulation_date': simulation_date,
#                 'expected_label': expected_label,
#                 'final_score': final_score,
#                 'predicted_label': predicted_label,
#                 'is_correct': is_correct,
#                 'status': status,
#                 'scenario': scenario,
#                 'decision_inputs': decision_inputs
#             })
            
#         except Exception as e:
#             print(f">>> Error: {str(e)}")
#             results.append({
#                 'id': hypothesis_id,
#                 'short_name': short_name,
#                 'expected_label': expected_label,
#                 'error': str(e),
#                 'is_correct': False
#             })
    
#     return results

In [13]:
# # Run Group A test (A1-A15, all high_novelty)
# # Modify start_idx and end_idx to test different ranges:
# #   - Group A (high_novelty): 0-15
# #   - Group B (incremental): 15-30
# #   - Group C (specious): 30-45

# results = await evaluate_batch(df, start_idx=0, end_idx=1, model="gpt-4o")

In [14]:
# # Display results summary
# def print_summary(results):
#     print("\n" + "#"*60)
#     print("# EVALUATION RESULTS SUMMARY")
#     print("#"*60)
    
#     successful = [r for r in results if 'error' not in r]
#     failed = [r for r in results if 'error' in r]
#     correct = [r for r in successful if r.get('is_correct', False)]
    
#     print(f"\nTotal: {len(results)}")
#     print(f"Successful: {len(successful)}, Failed: {len(failed)}")
#     print(f"Correct: {len(correct)}/{len(successful)} = {len(correct)/len(successful)*100:.1f}%")
    
#     # Per-category statistics
#     for category in ['high_novelty', 'incremental', 'specious']:
#         cat_results = [r for r in successful if r['expected_label'] == category]
#         if cat_results:
#             cat_correct = sum(1 for r in cat_results if r['is_correct'])
#             scores = [r['final_score'] for r in cat_results]
#             print(f"\n{category.upper()}:")
#             print(f"  Accuracy: {cat_correct}/{len(cat_results)} = {cat_correct/len(cat_results)*100:.1f}%")
#             print(f"  Mean Score: {sum(scores)/len(scores):.2f}")
#             print(f"  Score Range: [{min(scores):.1f}, {max(scores):.1f}]")
    
#     # Detailed results table
#     print("\n" + "-"*60)
#     print("Detailed Results:")
#     print("-"*60)
#     for r in results:
#         if 'error' in r:
#             print(f"❌ {r['id']:4} | {r.get('short_name', 'N/A'):15} | ERROR: {r['error'][:30]}")
#         else:
#             mark = '✅' if r['is_correct'] else '❌'
#             print(f"{mark} {r['id']:4} | {r['short_name']:15} | Score: {r['final_score']:4} | "
#                   f"Predicted: {r['predicted_label']:12} | Expected: {r['expected_label']}")
    
#     return results

# print_summary(results)

In [15]:
# # Save results to file
# output_dir = Path(PROJECT_ROOT) / 'playground' / 'evaluation_results'
# output_dir.mkdir(parents=True, exist_ok=True)

# # Save JSON
# with open(output_dir / 'notebook_results.json', 'w', encoding='utf-8') as f:
#     json.dump({
#         'timestamp': datetime.now().isoformat(),
#         'total': len(results),
#         'results': results
#     }, f, indent=2, ensure_ascii=False, default=str)

# # Save CSV summary
# results_df = pd.DataFrame([{
#     'ID': r.get('id'),
#     'Short Name': r.get('short_name'),
#     'Simulation Date': r.get('simulation_date'),
#     'Expected Label': r.get('expected_label'),
#     'Final Score': r.get('final_score'),
#     'Predicted Label': r.get('predicted_label'),
#     'Is Correct': r.get('is_correct'),
#     'Status': r.get('status'),
#     'Scenario': r.get('scenario'),
#     'Error': r.get('error')
# } for r in results])
# results_df.to_csv(output_dir / 'notebook_results.csv', index=False, encoding='utf-8')

# print(f"Results saved to: {output_dir}")